# 08 · Patient model: prepare, train, evaluate

**SimClinic, CS425 Group 7 (Marcus).** This notebook does everything for the patient model, from the cleaned data to the report's numbers. It imports nothing from `simclinic/`. The formatting code and Shiv's symptom matcher are copied into it, so it runs anywhere the data files are.

| input | from |
|---|---|
| `data/processed/diagnose-me/clean/{train,dev}.jsonl` | notebook 02: real patient messages (Stage 1) |
| `data/processed/dialogues/dialogues.jsonl` | Shiv, notebook 07: 5,229 card-conditioned consultations (Stage 2) |
| `data/processed/cards/cards.jsonl` | Shiv, notebook 07: 2,013 case cards (evaluation) |
| `simclinic/symptom_lexicon.json` | Shiv: lay phrasings for the header and the symptom matcher (read as data) |

| output | |
|---|---|
| `data/processed/patient/` | tokenized Stage 1 and Stage 2 data and the tokenizer |
| `models/patient/<run>/` | one fine-tuned model per run. `simclinic/patient.py`'s `PatientSimulator` loads it in the app. |
| `results/patient_runs.csv` | one row per training run |
| `results/patient_eval_<split>.csv`, `results/patient_eval_summary_<split>.csv` | evaluation per dialogue and per system |
| `results/rater_sheet_<split>.csv`, `results/rater_key_<split>.csv` | blinded transcripts for the two raters, and which system wrote each one |

## The plan
**Training: two stages.**
1. **Stage 1, patient voice.** Real diagnose-me patient messages teach GPT-2 how patients describe a complaint: informal, hedging, with typos. There is no case card.
2. **Stage 2, case-card role-play.** Training continues on Shiv's consultations, with the card in a hidden header. This teaches the model to stay on the card over many turns and never name the disease.

The grid follows the proposal: GPT-2 small and medium, LR ∈ {5e-5, 1e-4}, effective batch 16, early stopping on dev loss. Two ablation runs skip Stage 1 to show whether it helps. The run table is in section 3.

**Evaluation: auto mode with a scripted doctor.** Every system answers the same 16 SOCRATES history-taking questions, so they are compared on identical input.
- **symptom fidelity:** of the symptoms the patient says it has, the share that are on the card, matched with Shiv's lexicon and `on_card` rule
- **coverage:** the share of card symptoms the patient revealed
- **disease leakage:** the share of dialogues where the patient names its disease
- **staying in character:** 1–5, from two raters on blinded transcripts

The systems compared are the prompted-only pretrained GPT-2 (small and medium), the fine-tuned small and medium, and the no-Stage-1 ablation.

**Model selection** uses dev: Stage 2 dev loss plus the in-character score on 50 dev cards. The **report numbers** come from the 201 test cards.

## How to run it on the cluster
Copy the repo with the four inputs above, then `pip install -r requirements.txt`. Set the values in section 0, then run all cells.
1. **Job 1:** keep the defaults. This prepares the data, trains all 8 runs, and evaluates on dev. If the job is killed, run the notebook again: finished steps are skipped, and an unfinished run resumes from its last epoch.
   - **Optional:** to run the two Stage 1 runs as parallel jobs first, use two copies of the notebook, each with `STEPS = ["prepare", "train"]`. Set `RUNS = ["stage1-gpt2"]` in one copy and `RUNS = ["stage1-gpt2-medium"]` in the other. Let the prepare step finish in one copy before starting the other.
2. **Raters** score `results/rater_sheet_dev.csv`.
3. **Job 2:** set `STEPS = ["eval"]`, `EVAL_SPLIT = "test"`, and `FINAL_RUNS` to the chosen run per size, for example `["stage2-gpt2-lr5e-05", "stage2-gpt2-medium-lr5e-05"]`.

To run without opening Jupyter (for example in a batch job), use:
```bash
jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 notebooks/08_patient_model.ipynb --output-dir runs --output job1.ipynb
```
Run job 2 again once the raters have scored `results/rater_sheet_test.csv`. Their scores are read back in, and the sheet is never overwritten. Bring back `models/patient/<final runs>/` and `results/`.

`QUICK = True` runs every cell in a few minutes on a CPU, with tiny data, 1 epoch and 3 cards. It writes to `models/patient-quick/` and `results/quick/` only.

## 0. Parameters
Edit these values before running.

In [1]:
STEPS = ["eval"]   # any of these, run in this order
RUNS = None                 # None = every run in the plan (section 3), or a list of run names
EVAL_SPLIT = "test"          # "dev": 50 cards, for choosing the final runs; "test": all 201 cards, for the report
FINAL_RUNS = ["stage2-gpt2-lr0.0001", "stage2-gpt2-medium-lr5e-05"]           # test: the chosen Stage 2 run per size; None = the lowest dev loss per size
RATER_CARDS = 30            # test: cards per system in the rater sheet (dev uses all 50 cards)
BATCH = {"gpt2": 8, "gpt2-medium": 4}   # per-GPU batch; lower it if the GPU runs out of memory
QUICK = False               # tiny CPU check of every cell; writes to models/patient-quick and results/quick
SEED = 42

In [2]:
%pip install --upgrade "transformers>=5.0" "datasets>=4.0" "accelerate>=1.0" "pandas>=2.2"

Note: you may need to restart the kernel to use updated packages.


## 1. Setup

In [3]:
import gc, hashlib, json, math, random, re, shutil, time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch

pd.set_option("display.max_colwidth", 120)
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent

# --- inputs ------------------------------------------------------------------------------------
DIAGNOSE_ME_DIR = ROOT / "data" / "processed" / "diagnose-me" / "clean"    # notebook 02
DIALOGUES_JSONL = ROOT / "data" / "processed" / "dialogues" / "dialogues.jsonl"
CARDS_JSONL = ROOT / "data" / "processed" / "cards" / "cards.jsonl"
LEXICON_JSON = ROOT / "simclinic" / "symptom_lexicon.json"

# --- outputs -----------------------------------------------------------------------------------
PREP_DIR = ROOT / "data" / "processed" / ("patient-quick" if QUICK else "patient")
MODELS_DIR = ROOT / "models" / ("patient-quick" if QUICK else "patient")
RESULTS_DIR = ROOT / "results" / "quick" if QUICK else ROOT / "results"
RUNS_CSV = RESULTS_DIR / "patient_runs.csv"

# --- format and training -----------------------------------------------------------------------
BASE_TOKENIZER = "gpt2"      # gpt2 and gpt2-medium share it, so one tokenized dataset serves both
MAX_LENGTH = {1: 512, 2: 1024}
EFFECTIVE_BATCH = 16         # proposal: effective batch 16 through gradient accumulation
EPOCHS = {1: 2, 2: 5}        # Stage 1 has 217k examples, so 2 passes are plenty; proposal: 3-5 epochs for Stage 2
EARLY_STOPPING_PATIENCE = 1
WARMUP, WEIGHT_DECAY = 0.05, 0.01
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
FP16 = DEVICE == "cuda" and not BF16
N_GPUS = max(1, torch.cuda.device_count())

# --- evaluation --------------------------------------------------------------------------------
N_EVAL_CARDS = {"dev": 50, "test": None}   # None = every card in the split
GEN = {"temperature": 0.8, "top_p": 0.9, "max_new_tokens": 80}
GEN_BATCH = 32 if DEVICE == "cuda" else 4
RATERS = ["rater_1", "rater_2"]

if QUICK:   # a few minutes on a CPU: small data, 1 epoch, 3 cards, 4 questions
    QUICK_ROWS = {"train": 64, "dev": 16}
    EPOCHS = {1: 1, 2: 1}
    N_EVAL_CARDS = {"dev": 3, "test": 3}
    GEN["max_new_tokens"] = 30

for p in (DIAGNOSE_ME_DIR / "train.jsonl", DIALOGUES_JSONL, CARDS_JSONL, LEXICON_JSON):
    assert p.exists(), f"missing input: {p}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"device {DEVICE} x{N_GPUS} (bf16 {BF16}, fp16 {FP16}) | steps {STEPS} | quick {QUICK}")

device cuda x1 (bf16 True, fp16 False) | steps ['eval'] | quick False


## 2. The training format
This is the same format as `simclinic/patient.py`, which the app uses at inference. Section 6 checks that the two give identical strings.
```
<|disease|>Flu<|symptoms|>a fever; a cough<|persona|>20, M, Malay, software engineer, anxious, medium, this morning, moderate
<|doctor|>What brings you in today?<|patient|>I've had a fever since this morning...<|doctor|>Any cough?<|patient|>Yeah, a dry one.<|doctor|>
└──────────────────────── hidden header: masked ─────────────────────┘                └── loss ──────────────────────────┘         └── loss ─────────┘
```
- The header lists symptoms in lay words: the first phrasing in Shiv's lexicon, so "Coryza" becomes "a runny nose".
- The loss covers every patient turn and the `<|doctor|>` after it, so the model learns where its turn ends. Generation stops at that token.
- Stage 1 has no card, so its strings have no header.

In [4]:
ROLE_TOKENS = {"doctor": "<|doctor|>", "patient": "<|patient|>"}
HEADER_TOKENS = {"disease": "<|disease|>", "symptoms": "<|symptoms|>", "persona": "<|persona|>"}
DIAGNOSIS_TOKEN = "<|diagnosis|>"        # used by the doctor model; added so both models share token ids
SPECIAL_TOKENS = list(ROLE_TOKENS.values()) + list(HEADER_TOKENS.values()) + [DIAGNOSIS_TOKEN]
STOP_TOKEN = ROLE_TOKENS["doctor"]
DOCTOR_OPENER = "What brings you in today?"
PERSONA_FIELDS = ["age", "sex", "ethnicity", "occupation", "temperament", "health_literacy", "onset", "severity"]
LAY = {s: p[0] for s, p in json.loads(LEXICON_JSON.read_text(encoding="utf-8")).items() if not s.startswith("_") and p}

def lay(symptom):
    return LAY.get(symptom, symptom.lower())

def format_header(card=None):
    if not card:
        return ""
    persona = card.get("persona") or {}
    persona_text = ", ".join(str(persona[f]) for f in PERSONA_FIELDS if persona.get(f) not in (None, ""))
    return (HEADER_TOKENS["disease"] + card["disease"]
            + HEADER_TOKENS["symptoms"] + "; ".join(lay(s) for s in card["symptoms"])
            + HEADER_TOKENS["persona"] + persona_text)

def with_opener(turns, opener=DOCTOR_OPENER):
    turns = list(turns)
    if opener and (not turns or turns[0]["role"] != "doctor"):
        turns = [{"role": "doctor", "text": opener}] + turns
    return turns

def format_dialogue(turns, card=None):
    """Training string and the [start, end) character spans that carry loss. Turns after the last
    patient turn are dropped; the string ends with the <|doctor|> that closes the last patient turn."""
    turns = with_opener(turns)
    roles = [t["role"] for t in turns]
    assert all(a != b for a, b in zip(roles, roles[1:])), f"roles must alternate: {roles}"
    turns = turns[: max(i for i, r in enumerate(roles) if r == "patient") + 1]
    text, spans = format_header(card), []
    for t in turns:
        text += ROLE_TOKENS[t["role"]]
        start = len(text)
        text += t["text"]
        if t["role"] == "patient":
            spans.append((start, len(text) + len(STOP_TOKEN)))
    return text + STOP_TOKEN, spans

def mask_labels(input_ids, offsets, spans):
    """labels = input_ids, with -100 on every token that does not start inside a span."""
    return [tok if (e > s and any(a <= s < b for a, b in spans)) else -100
            for tok, (s, e) in zip(input_ids, offsets)]

def build_prompt(card, history, tokenizer, max_len):
    """Inference prompt: the header plus the most recent turns that fit, ending in <|patient|>.
    The header is never dropped, and the window always starts with a doctor turn."""
    turns = with_opener(history)
    header = format_header(card)
    render = lambda ts: header + "".join(ROLE_TOKENS[t["role"]] + t["text"] for t in ts) + ROLE_TOKENS["patient"]
    while len(turns) > 1 and len(tokenizer(render(turns))["input_ids"]) > max_len:
        turns = turns[1:]
        while turns[0]["role"] != "doctor":
            turns = turns[1:]
    return render(turns)

print(format_header({"disease": "Flu", "symptoms": ["Fever", "Coryza"],
                     "persona": {"age": 20, "sex": "M", "ethnicity": "Malay", "occupation": "software engineer"}}))

<|disease|>Flu<|symptoms|>a fever; a runny nose<|persona|>20, M, Malay, software engineer


## 3. Prepare the data
This step is skipped when `data/processed/patient/prep_config.json` exists. Delete the folder to rebuild.
- **Stage 1:** each diagnose-me patient message (notebook 02's split) becomes `<|doctor|>What brings you in today?<|patient|>…<|doctor|>`, truncated at 512 tokens. This is the same string notebook 02 tokenized.
- **Stage 2:** Shiv's dialogues, with each record's card and Shiv's split (by card, stratified by disease). A dialogue longer than 1,024 tokens drops its oldest turns until it fits. The header is never dropped, which is the same rule `build_prompt` uses at inference.

In [5]:
from datasets import Dataset, DatasetDict, load_from_disk
from transformers import AutoTokenizer

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

def make_tokenizer():
    tok = AutoTokenizer.from_pretrained(BASE_TOKENIZER)
    tok.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})
    tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    return tok

def encode_stage1(batch, tokenizer):
    texts, spans = zip(*(format_dialogue([{"role": "patient", "text": p}]) for p in batch["patient"]))
    enc = tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH[1], return_offsets_mapping=True)
    enc["labels"] = [mask_labels(ids, offs, sp) for ids, offs, sp in zip(enc["input_ids"], enc.pop("offset_mapping"), spans)]
    return enc

def encode_stage2(turns, card, tokenizer):
    """One Stage 2 example. turns_dropped counts the oldest turns removed to fit MAX_LENGTH[2]."""
    while turns[-1]["role"] != "patient":            # trailing doctor turns are never trained
        turns = turns[:-1]
    dropped = 0
    while True:
        text, spans = format_dialogue(turns, card)
        enc = tokenizer(text, return_offsets_mapping=True)
        if len(enc["input_ids"]) <= MAX_LENGTH[2] or len(turns) <= 2:
            break
        turns, dropped = turns[1:], dropped + 1
        while turns[0]["role"] != "doctor":
            turns, dropped = turns[1:], dropped + 1
    truncated = len(enc["input_ids"]) > MAX_LENGTH[2]
    if truncated:
        enc = tokenizer(text, truncation=True, max_length=MAX_LENGTH[2], return_offsets_mapping=True)
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"],
            "labels": mask_labels(enc["input_ids"], enc["offset_mapping"], spans),
            "turns_dropped": dropped, "truncated": truncated, "text": text}

def trained_text(ex, tokenizer):
    return tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])

def prepare():
    tokenizer = make_tokenizer()
    # --- Stage 1: diagnose-me patient messages ---
    stage1 = {}
    for split in ("train", "dev"):
        rows = read_jsonl(DIAGNOSE_ME_DIR / f"{split}.jsonl")
        if QUICK:
            rows = rows[: QUICK_ROWS[split] * 4]
        stage1[split] = Dataset.from_dict({"id": [r["dialogue_id"] for r in rows],
                                           "patient": [r["turns"][0]["text"] for r in rows]})
        assert all(r["turns"][0]["role"] == "patient" for r in rows)
    stage1 = DatasetDict(stage1).map(encode_stage1, batched=True, batch_size=1000,
                                     fn_kwargs={"tokenizer": tokenizer}, desc="stage 1")
    for split in stage1:   # a sample: the trained tokens are exactly the patient message plus <|doctor|>
        for ex in stage1[split].select(range(min(1000, len(stage1[split])))):
            if len(ex["input_ids"]) < MAX_LENGTH[1]:
                assert trained_text(ex, tokenizer) == ex["patient"] + STOP_TOKEN, ex["id"]
    stage1 = stage1.remove_columns(["patient"])

    # --- Stage 2: Shiv's consultations ---
    dialogues = read_jsonl(DIALOGUES_JSONL)
    by_card = {}
    for d in dialogues:
        by_card.setdefault(d["card_id"], set()).add(d["split"])
        assert all(t["role"] == ("doctor" if i % 2 == 0 else "patient") for i, t in enumerate(d["turns"])), d["dialogue_id"]
        assert set(d["persona"]) == set(PERSONA_FIELDS), d["dialogue_id"]
    assert all(len(s) == 1 for s in by_card.values()), "a card is in two splits"
    diseases = {d["disease"] for d in dialogues}
    for split in ("train", "dev", "test"):
        assert {d["disease"] for d in dialogues if d["split"] == split} == diseases, f"a disease is missing from {split}"
    rows = {"train": [], "dev": [], "test": []}
    for d in dialogues:
        card = {"disease": d["disease"], "symptoms": d["symptoms"], "persona": d["persona"]}
        rows[d["split"]].append({"dialogue_id": d["dialogue_id"], "card_id": d["card_id"], **encode_stage2(d["turns"], card, tokenizer)})
    if QUICK:
        rows = {s: r[: QUICK_ROWS.get(s, 16) * 2] for s, r in rows.items()}
    stage2 = DatasetDict({s: Dataset.from_list(r) for s, r in rows.items()})
    by_id = {d["dialogue_id"]: d for d in dialogues}
    for split in stage2:   # every example: header masked, trained tokens are exactly the patient turns
        for ex in stage2[split]:
            d = by_id[ex["dialogue_id"]]
            window = d["turns"][ex["turns_dropped"]:]
            expected = "".join(t["text"] + STOP_TOKEN for t in window if t["role"] == "patient")
            if not ex["truncated"]:
                assert trained_text(ex, tokenizer) == expected, ex["dialogue_id"]

    def stats(ds):
        n = pd.Series([len(x) for x in ds["input_ids"]], dtype=float)
        out = {"examples": len(ds), "tokens p50": n.median(), "tokens p95": n.quantile(.95), "tokens max": n.max(),
               "trainable tokens": sum(l != -100 for lab in ds["labels"] for l in lab)}
        if "turns_dropped" in ds.column_names:
            out |= {"turns dropped": sum(x > 0 for x in ds["turns_dropped"]), "truncated": sum(ds["truncated"])}
        return out
    table = pd.DataFrame({f"stage {i} {s}": stats(ds[s]) for i, ds in ((1, stage1), (2, stage2)) for s in ds}).T

    PREP_DIR.mkdir(parents=True, exist_ok=True)
    stage1.save_to_disk(str(PREP_DIR / "stage1"))
    stage2.remove_columns(["turns_dropped", "truncated", "text"]).save_to_disk(str(PREP_DIR / "stage2"))
    tokenizer.save_pretrained(str(PREP_DIR / "tokenizer"))
    (PREP_DIR / "prep_config.json").write_text(json.dumps({
        "inputs": [str(p.relative_to(ROOT)) for p in (DIAGNOSE_ME_DIR, DIALOGUES_JSONL, LEXICON_JSON)],
        "tokenizer": BASE_TOKENIZER, "special_tokens": SPECIAL_TOKENS, "vocab_size": len(tokenizer),
        "max_length": MAX_LENGTH, "doctor_opener": DOCTOR_OPENER, "persona_fields": PERSONA_FIELDS, "quick": QUICK,
        "counts": table["examples"].astype(int).to_dict()}, indent=2), encoding="utf-8")
    return table, stage2

if "prepare" in STEPS and not (PREP_DIR / "prep_config.json").exists():
    prep_table, stage2_preview = prepare()
    display(prep_table.round(0))
    ex = stage2_preview["train"][0]
    tok = AutoTokenizer.from_pretrained(str(PREP_DIR / "tokenizer"))
    print("one Stage 2 example (trained tokens in «»):")
    print("".join(f"«{tok.decode([t])}»" if l != -100 else tok.decode([t]) for t, l in list(zip(ex["input_ids"], ex["labels"]))[:220]))
else:
    print("data already prepared:", json.loads((PREP_DIR / "prep_config.json").read_text())["counts"] if (PREP_DIR / "prep_config.json").exists() else "not yet")

/common/home/projectgrps/CS425/CS425G7/jupyterlab-venv-pytorch-240/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data already prepared: {'stage 1 train': 217214, 'stage 1 dev': 12002, 'stage 2 train': 4827, 'stage 2 dev': 201, 'stage 2 test': 201}


## 4. Train
| run | starts from | data | LR |
|---|---|---|---|
| `stage1-gpt2`, `stage1-gpt2-medium` | pretrained GPT-2 | diagnose-me | 5e-5 |
| `stage2-gpt2-lr5e-05`, `stage2-gpt2-lr0.0001` | `stage1-gpt2` | Shiv's dialogues | 5e-5, 1e-4 |
| `stage2-gpt2-medium-lr5e-05`, `stage2-gpt2-medium-lr0.0001` | `stage1-gpt2-medium` | Shiv's dialogues | 5e-5, 1e-4 |
| `stage2-gpt2-lr5e-05-nostage1`, `stage2-gpt2-medium-lr5e-05-nostage1` | pretrained GPT-2 (ablation) | Shiv's dialogues | 5e-5 |

**Training setup**
- **Batch and precision:** effective batch 16 from per-GPU batch × GPUs × gradient accumulation; bf16 where the GPU supports it, otherwise fp16.
- **Epochs:** up to 2 for Stage 1 and 5 for Stage 2. Training stops once dev loss has not improved for one epoch, and the best epoch's weights are kept.
- **Batching:** `group_by_length` puts examples of similar length together, which cuts padding.
- **Reference loss:** dev loss before training is shown for reference only. It takes no part in early stopping.

**Running and resuming**
- **Order:** runs go in table order.
- **Finished runs:** a finished run has `train_config.json` and is skipped.
- **Interrupted runs:** an interrupted run resumes from its last saved epoch.
- **Dependencies:** a Stage 2 run waits for its Stage 1 run.
- **Cost:** Stage 1 is the expensive part, at about 13.6k optimizer steps per epoch. Stage 2 is about 300 steps per epoch.

In [6]:
from transformers import (AutoModelForCausalLM, DataCollatorForSeq2Seq, EarlyStoppingCallback, Trainer,
                          TrainingArguments)
from transformers.trainer_utils import get_last_checkpoint

RUN_PLAN = {
    "stage1-gpt2":                         {"stage": 1, "model": "gpt2",        "init": "gpt2",               "lr": 5e-5},
    "stage1-gpt2-medium":                  {"stage": 1, "model": "gpt2-medium", "init": "gpt2-medium",        "lr": 5e-5},
    "stage2-gpt2-lr5e-05":                 {"stage": 2, "model": "gpt2",        "init": "stage1-gpt2",        "lr": 5e-5},
    "stage2-gpt2-lr0.0001":                {"stage": 2, "model": "gpt2",        "init": "stage1-gpt2",        "lr": 1e-4},
    "stage2-gpt2-medium-lr5e-05":          {"stage": 2, "model": "gpt2-medium", "init": "stage1-gpt2-medium", "lr": 5e-5},
    "stage2-gpt2-medium-lr0.0001":         {"stage": 2, "model": "gpt2-medium", "init": "stage1-gpt2-medium", "lr": 1e-4},
    "stage2-gpt2-lr5e-05-nostage1":        {"stage": 2, "model": "gpt2",        "init": "gpt2",               "lr": 5e-5},
    "stage2-gpt2-medium-lr5e-05-nostage1": {"stage": 2, "model": "gpt2-medium", "init": "gpt2-medium",        "lr": 5e-5},
}
QUICK_RUNS = ["stage1-gpt2", "stage2-gpt2-lr5e-05"]

def finished(run):
    return (MODELS_DIR / run / "train_config.json").exists()

def read_runs():
    return pd.read_csv(RUNS_CSV).drop_duplicates("run", keep="last") if RUNS_CSV.exists() else pd.DataFrame()

def train(run):
    spec = RUN_PLAN[run]
    out_dir, stage = MODELS_DIR / run, spec["stage"]
    tokenizer = AutoTokenizer.from_pretrained(str(PREP_DIR / "tokenizer"))
    data = load_from_disk(str(PREP_DIR / f"stage{stage}")).select_columns(["input_ids", "attention_mask", "labels"])
    train_ds, dev_ds = data["train"].shuffle(seed=SEED), data["dev"].shuffle(seed=SEED)
    if QUICK:
        train_ds, dev_ds = train_ds.select(range(QUICK_ROWS["train"])), dev_ds.select(range(QUICK_ROWS["dev"]))

    init = MODELS_DIR / spec["init"] if spec["init"] in RUN_PLAN else spec["init"]
    torch.manual_seed(SEED)
    model = AutoModelForCausalLM.from_pretrained(str(init))
    if model.get_input_embeddings().num_embeddings != len(tokenizer):
        model.resize_token_embeddings(len(tokenizer))      # pretrained GPT-2: rows for the special tokens
    model.config.pad_token_id = tokenizer.pad_token_id

    per_device = BATCH[spec["model"]]
    args = TrainingArguments(
        output_dir=str(out_dir / "checkpoints"), learning_rate=spec["lr"], num_train_epochs=EPOCHS[stage],
        warmup_steps=WARMUP, weight_decay=WEIGHT_DECAY, per_device_train_batch_size=per_device,
        per_device_eval_batch_size=per_device, gradient_accumulation_steps=max(1, EFFECTIVE_BATCH // (per_device * N_GPUS)),
        bf16=BF16, fp16=FP16, eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        train_sampling_strategy="group_by_length", logging_steps=0.05, disable_tqdm=True, report_to="none", seed=SEED)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds, processing_class=tokenizer,
                      data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)])
    last = get_last_checkpoint(args.output_dir) if Path(args.output_dir).exists() else None
    print(f"=== {run}: {len(train_ds):,} train / {len(dev_ds):,} dev, from {spec['init']}" + (f", resuming {Path(last).name}" if last else ""))
    baseline = trainer.evaluate()["eval_loss"]             # before training, for reference only
    t0 = time.time()
    trainer.train(resume_from_checkpoint=last)
    minutes = (time.time() - t0) / 60

    evals = pd.DataFrame([h for h in trainer.state.log_history if "eval_loss" in h and h.get("epoch", 0) > 0])[["epoch", "eval_loss"]]
    best = evals.loc[evals["eval_loss"].idxmin()]
    trainer.save_model(str(out_dir))                       # the best epoch: load_best_model_at_end reloaded it
    row = {"run": run, "stage": stage, "model": spec["model"], "init_from": spec["init"], "lr": spec["lr"],
           "train_examples": len(train_ds), "dev_examples": len(dev_ds), "dev_loss_before": round(baseline, 4),
           "epochs_run": round(trainer.state.epoch, 2), "best_epoch": float(best["epoch"]),
           "best_dev_loss": round(float(best["eval_loss"]), 4), "dev_ppl": round(math.exp(best["eval_loss"]), 2),
           "minutes": round(minutes, 1), "device": f"{DEVICE} x{N_GPUS}", "date": datetime.now().strftime("%Y-%m-%d %H:%M")}
    pd.DataFrame([row]).to_csv(RUNS_CSV, mode="a", header=not RUNS_CSV.exists(), index=False)
    (out_dir / "train_config.json").write_text(json.dumps({**row, "epoch_losses": evals.to_dict("records")}, indent=2), encoding="utf-8")
    shutil.rmtree(out_dir / "checkpoints", ignore_errors=True)
    print(f"{run}: dev loss {baseline:.3f} before -> {best['eval_loss']:.3f} at epoch {best['epoch']:g} ({minutes:.1f} min)")
    del trainer, model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

if "train" in STEPS:
    for run in RUNS or (QUICK_RUNS if QUICK else list(RUN_PLAN)):
        init = RUN_PLAN[run]["init"]
        if finished(run):
            print(f"{run}: already trained, skipped")
        elif init in RUN_PLAN and not finished(init):
            print(f"{run}: skipped, needs {init} first")
        else:
            train(run)
read_runs()

,run,stage,model,init_from,lr,train_examples,dev_examples,dev_loss_before,epochs_run,best_epoch,best_dev_loss,dev_ppl,minutes,device,date
0,stage1-gpt2,1,gpt2,gpt2,0.00005,217214,12002,4.3024,2.0,2.0,3.1589,23.54,38.5,cuda x1,2026-11-01 05:42
1,stage1-gpt2-medium,1,gpt2-medium,gpt2-medium,0.00005,217214,12002,4.0663,2.0,2.0,2.9904,19.89,119.9,cuda x1,2026-11-01 07:43
2,stage2-gpt2-lr5e-05,2,gpt2,stage1-gpt2,0.00005,4827,201,2.2633,5.0,5.0,0.9258,2.52,3.5,cuda x1,2026-11-01 07:47
3,stage2-gpt2-lr0.0001,2,gpt2,stage1-gpt2,0.00010,4827,201,2.2633,5.0,5.0,0.8876,2.43,3.5,cuda x1,2026-11-01 07:50
4,stage2-gpt2-medium-lr5e-05,2,gpt2-medium,stage1-gpt2-medium,0.00005,4827,201,2.2721,5.0,5.0,0.8242,2.28,9.6,cuda x1,2026-11-01 08:00
5,stage2-gpt2-medium-lr0.0001,2,gpt2-medium,stage1-gpt2-medium,0.00010,4827,201,2.2721,5.0,5.0,0.8513,2.34,9.6,cuda x1,2026-11-01 08:10
6,stage2-gpt2-lr5e-05-nostage1,2,gpt2,gpt2,0.00005,4827,201,3.7570,5.0,5.0,0.9325,2.54,3.5,cuda x1,2026-11-01 08:13
7,stage2-gpt2-medium-lr5e-05-nostage1,2,gpt2-medium,gpt2-medium,0.00005,4827,201,3.6662,5.0,5.0,0.8358,2.31,9.5,cuda x1,2026-11-01 08:23


## 5. Evaluate
**The interviewer.** Every system answers the same 16 doctor questions, which follow the SOCRATES checklist and then the standard history questions. None asks what the patient thinks the illness is, so naming the disease always counts as leakage.

**Generation.** All cards are interviewed together, one question at a time, in batches. Sampling uses temperature 0.8 and top-p 0.9 and stops at `<|doctor|>`.

**The prompted baseline** (pretrained GPT-2, no fine-tuning) gets the card in plain words above a `Doctor:` / `Patient:` transcript, and stops at the end of the line.

**Cards.**
- dev: 50 cards, sampled with the seed
- test: all 201 cards

**Systems.**
- dev: the prompted baselines and every finished Stage 2 run
- test: the prompted baselines, `FINAL_RUNS`, and the no-Stage-1 ablations

In [7]:
QUESTIONS = [
    "What brings you in today?",
    "Where exactly do you feel it?",                                   # Site
    "When did it start, and did it come on suddenly or gradually?",    # Onset
    "Can you describe what it feels like?",                            # Character
    "Does it spread anywhere else?",                                   # Radiation
    "Have you noticed any other symptoms?",                            # Associated symptoms
    "Is it there all the time, or does it come and go?",               # Time course
    "Does anything make it better or worse?",                          # Exacerbating / relieving
    "On a scale of 0 to 10, how bad is it?",                           # Severity
    "Have you had anything like this before?",
    "Are you taking any medication, or have you tried anything for it?",
    "Do you have any allergies?",
    "Do you have any other medical conditions?",
    "Does anyone in your family have similar problems?",
    "Do you smoke or drink alcohol?",
    "Is there anything else you'd like to tell me?",
]
if QUICK:
    QUESTIONS = QUESTIONS[:4]

class Patient:
    """One system. reply() writes the patient's next turn for a batch of consultations."""

    def __init__(self, source, prompted):
        self.prompted = prompted
        self.tokenizer = AutoTokenizer.from_pretrained(str(source))
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"
        self.model = AutoModelForCausalLM.from_pretrained(str(source)).to(DEVICE).eval()
        self.max_len = self.model.config.n_positions - GEN["max_new_tokens"]
        stop = "\n" if prompted else STOP_TOKEN
        self.stop_ids = [self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(stop))[0], self.tokenizer.eos_token_id]

    def prompt(self, card, history):
        if not self.prompted:
            return build_prompt(card, history, self.tokenizer, self.max_len)
        persona = ", ".join(f"{k.replace('_', ' ')}: {v}" for k, v in card["persona"].items())
        intro = (f"The following is a conversation between a doctor and a patient at a clinic. The patient ({persona}) "
                 f"has {card['disease']} but does not know the diagnosis and never names it. "
                 f"The patient's symptoms are: {', '.join(lay(s) for s in card['symptoms'])}.\n\n")
        lines = [("Doctor: " if t["role"] == "doctor" else "Patient: ") + t["text"] for t in history]
        while len(lines) > 1 and len(self.tokenizer(intro + "\n".join(lines) + "\nPatient:")["input_ids"]) > self.max_len:
            lines = lines[1:]
            while not lines[0].startswith("Doctor: "):
                lines = lines[1:]
        return intro + "\n".join(lines) + "\nPatient:"

    @torch.no_grad()
    def reply(self, cards, histories):
        enc = self.tokenizer([self.prompt(c, h) for c, h in zip(cards, histories)], return_tensors="pt", padding=True).to(DEVICE)
        out = self.model.generate(**enc, do_sample=True, temperature=GEN["temperature"], top_p=GEN["top_p"],
                                  max_new_tokens=GEN["max_new_tokens"], eos_token_id=self.stop_ids,
                                  pad_token_id=self.tokenizer.eos_token_id)
        replies = []
        for text in self.tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:]):
            for cut in (["\n", "Doctor:"] if self.prompted else SPECIAL_TOKENS) + [self.tokenizer.eos_token]:
                text = text.split(cut)[0]
            replies.append(text.strip())
        return replies

def interview(system, cards_):
    """Run the scripted interview with every card at once. Returns one transcript per card."""
    histories = [[] for _ in cards_]
    for question in QUESTIONS:
        for h in histories:
            h.append({"role": "doctor", "text": question})
        for i in range(0, len(cards_), GEN_BATCH):
            for h, r in zip(histories[i:i + GEN_BATCH], system.reply(cards_[i:i + GEN_BATCH], histories[i:i + GEN_BATCH])):
                h.append({"role": "patient", "text": r})
    return histories

### Shiv's symptom matcher (copied verbatim)
This is the same matcher Shiv's dialogue filter uses, so symptom fidelity here means exactly what "on the card" meant when the training data was filtered. It was copied from `simclinic/lexicon.py` with one change, `DEFAULT_PATH`. **If Shiv changes `lexicon.py`, rebuild this cell from it.**

In [8]:
# ---- copied verbatim from simclinic/lexicon.py ----
"""Symptom lexicon: Symcat names <-> lay phrasings, and extraction of revealed symptoms from a dialogue.

`revealed_symptoms()` is what the naive Bayes baseline reads: for every symptom the dialogue mentions it
decides whether the patient has it (True) or denied it (False). It combines two cues:
  * explicit mentions in patient turns, with a simple clause-level negation check
    ("I've had a cough but no fever" -> cough True, fever False);
  * yes/no answers to the doctor's question ("Any fever?" -> "Nope." -> fever False).

Matching is more forgiving than the phrase list alone:
  * "the" and "your" read as "my", so "pain around the bottom" also matches "pain around my bottom";
  * degree words are dropped before matching ("my throat is really sore" -> "my throat is sore");
  * every "<state> <part>" phrase also matches as "<part> is / feels / looks <state>" ("sore throat" ->
    "throat feels sore"), and every "<part> pain" phrase as "<part> hurts", "pain in my <part>", ... .
    Listed phrases always win over these generated variants.

`on_card()` is the symptom-fidelity test: a mention is on-card when the card has the symptom, or one that
everyday speech does not tell apart from it ("I feel hot" on a card with fever).
"""
import json
import re
from functools import lru_cache
from pathlib import Path

DEFAULT_PATH = LEXICON_JSON                  # was: next to lexicon.py
SUFFIXES = ("ing", "es", "ed", "s")          # a phrase also matches with these endings ("rash" -> "rashes")

NEGATION = re.compile(r"\b(?:no|not|never|without|nor|neither|none|deny|denies)\b|n't\b")
UNCERTAIN = re.compile(r"\b(?:not sure|unsure|don't know|perhaps|might)\b")
# Clause boundaries for negation scope. A comma only ends a clause when a new subject follows
# ("I don't have a fever, I have a cough"), so "no fever, cough or headache" stays one clause.
# "just" / "only" start the part a patient does have ("no chest pain, just the stomach").
CLAUSE_BREAK = re.compile(r"[.;!?]|\b(?:but|though|although|however|except|apart from|besides|other than"
                          r"|just|only)\b"
                          r"|,\s*(?=(?:i|i've|i'm|it|it's|my|she|he|they|we)\b)"
                          r"|\b(?:and|so)\s+(?=(?:i|i've|i'm|it|it's|my)\b)")
DETERMINER = re.compile(r"\b(?:the|your)\b")
# Degree words carry no symptom information. "not really" is an answer, and "so I ..." starts a clause.
FILLER = re.compile(r"\b(?:a little bit|a little|a bit of|a bit|kind of|sort of|kinda|(?<!not )really|very|quite"
                    r"|pretty|extremely|terribly|super|slightly|somewhat|also|still|always|constantly|sometimes"
                    r"|started|starting|began|keeps|kept|so(?!\s+(?:i|i've|i'm|it|it's|my)\b))\b\s*")

# Spoken variants of "<state> <part>" and "<part> pain" phrases.
STATES = set("sore swollen red itchy stiff weak blocked stuffy stuffed dry painful numb puffy tight runny bloated "
             "flaky irritated watery sticky crusty bloodshot scaly cracked rough raw congested clogged plugged blurry "
             "blurred cloudy loose hard black dark pale bloody smelly pink discoloured discolored hoarse raspy croaky "
             "noisy irregular poor low reduced muffled yellow thick brittle scratchy achy aching tender inflamed "
             "enlarged fast rapid racing pounding thinning".split())
COPULAS = ["is", "are", "feels", "feel", "looks", "look", "has been", "have been", "is feeling", "are feeling",
           "gets", "get", "got", "was", "were", "seems", "seem"]
PAIN_PARTS = set("ankle arm back ear elbow eye face foot toe hand finger head hip joint knee leg mouth muscle neck "
                 "shoulder side sinus stomach tummy belly testicle tooth wrist chest throat body flank bladder pelvis "
                 "forehead".split()) | {"big toe", "lower back", "lower stomach", "upper stomach", "lower belly",
                                        "upper belly", "lower tummy"}
NOT_A_PART = set("a an the my in on of at to from around behind when while during or with".split())
PLURALS = {"foot": "feet", "tooth": "teeth"}
# "my eyes are red and swollen" -> "my eyes are red and eyes are swollen", so both states match.
COORDINATED = re.compile(rf"\b([a-z']+) ({'|'.join(sorted(COPULAS, key=len, reverse=True))}) ([a-z]+),? and ([a-z]+)\b")


def _distribute(m) -> str:
    part, copula, a, b = m.groups()
    return f"{part} {copula} {a} and {part} {copula} {b}" if a in STATES and b in STATES else m.group()
# A leading yes/no only counts as a separate answer when punctuation follows it: "No, I've had a cough"
# (cough present) but "No fever" (fever denied).
LEADING_ANSWER = re.compile(r"^\s*(?:no|nope|nah|not really|yes|yeah|yep|yup)\b\s*[,.!-]+\s*")
NO_ANSWER = re.compile(r"^\s*(?:no|nope|nah|not really|not at all|i don't|i haven't|i didn't|i'm not|never|none|nothing)\b")
YES_ANSWER = re.compile(r"^\s*(?:yes|yeah|yep|yup|yea|i do|i have|i've|i did|i am|definitely|a bit|a little|"
                        r"sometimes|quite|correct|sure|actually,? yes)\b")


def _normalise(text: str) -> str:
    # Questions say "pain in your ear", the lexicon says "pain in the ear", patients say "pain in my ear".
    text = text.lower().replace("’", "'").replace("‘", "'")
    return re.sub(r"\byou\b", "i", DETERMINER.sub("my", text))


def _simplify(text: str) -> str:
    """The text symptoms are matched in: normalised, without degree words, with coordinated states
    distributed. Idempotent, so offsets into it agree wherever it is recomputed."""
    return COORDINATED.sub(_distribute, re.sub(r"  +", " ", FILLER.sub("", _normalise(text))))


def _key(phrase: str) -> str:
    return " ".join(re.split(r"[\s-]+", _simplify(phrase).strip()))


def _plural(part: str) -> str:
    *head, last = part.split()
    return " ".join([*head, PLURALS.get(last, last if last.endswith("s") else last + "s")])


def spoken_variants(phrase: str) -> list:
    """How patients say a listed phrase in a sentence: "swollen lip" -> "my lip is swollen", "lips feel swollen";
    "ear pain" -> "my ear hurts", "ears are hurting", "pain in my ears"."""
    words = phrase.lower().split()
    if words and words[0] in ("a", "an"):
        words = words[1:]
    out = []
    if 2 <= len(words) <= 4 and words[0] in STATES and words[-1] != "pain" and not NOT_A_PART & set(words[1:]):
        state = words[0]
        for part in dict.fromkeys([" ".join(words[1:]), _plural(" ".join(words[1:]))]):
            out += [f"{part} {c} {state}" for c in COPULAS] + [f"{part} {state}"]    # "is my lip swollen?"
    if words[-1:] == ["pain"] and " ".join(words[:-1]) in PAIN_PARTS:
        for part in dict.fromkeys([" ".join(words[:-1]), _plural(" ".join(words[:-1]))]):
            out += [f"{part} hurt", f"{part} is hurt", f"{part} are hurt", f"{part} ache", f"{part} aching",
                    f"pain in my {part}", f"pains in my {part}", f"{part} is killing me"]
            out += [f"{part} {c} {s}" for c in ("has been", "have been", "was", "were") for s in ("hurt", "aching")]
            out += [f"{part} {c} {s}" for c in COPULAS for s in ("sore", "painful")]
    if words[-1:] == ["swelling"] and len(words) <= 3:
        part = " ".join(words[:-1])
        out += [f"swelling {p} my {part}" for p in ("in", "of", "around", "on")] + [f"{part} has swollen", f"{part} swelled"]
    return out


def _trie_regex(keys) -> str:
    """One regex for all phrases, built as a character trie so most positions fail on the first character.
    At every node the longer continuation is tried before ending, so the longest phrase wins
    ("lower back pain" over "back pain", "throwing up blood" over "throwing up")."""
    trie = {}
    for k in keys:
        node = trie
        for ch in k:
            node = node.setdefault(ch, {})
        node[""] = True

    def build(node, last):
        end = r"(?:ing|es|ed|s)?" if last.isalpha() else ""
        alts = [(r"[\s-]+" if ch == " " else re.escape(ch)) + build(child, ch)
                for ch, child in sorted(node.items()) if ch]
        if not alts:
            return end
        body = alts[0] if len(alts) == 1 else "(?:" + "|".join(alts) + ")"
        return f"(?:{body}|{end})" if "" in node else body

    return r"(?<![a-z'])" + build(trie, "") + r"(?![a-z])"


class Lexicon:
    def __init__(self, phrases: dict, symptoms=None):
        self.phrases = {s: list(p) for s, p in phrases.items() if not s.startswith("_")}
        for s in symptoms or ():
            self.phrases.setdefault(s, [s.lower()])          # fallback: the Symcat name itself
        # Stop phrases match like symptoms but stand for none, so they block a shorter phrase inside them:
        # "anxious about the results" is the persona's temperament, not the symptom "anxious".
        self._lookup = dict.fromkeys(map(_key, phrases.get("_not_symptoms", [])), "")
        for s, ps in self.phrases.items():
            for p in [*ps, s]:
                self._lookup.setdefault(_key(p), s)           # first symptom to claim a phrase keeps it
        for s, ps in self.phrases.items():                    # generated variants never displace a listed phrase
            for p in ps:
                for v in spoken_variants(p):
                    self._lookup.setdefault(_key(v), s)
        self.covered_by = {}                                  # symptom -> card symptoms that make it on-card
        for group in phrases.get("_same_complaint", []):
            for s in group:
                self.covered_by.setdefault(s, set()).update(g for g in group if g != s)
        for s, covering in phrases.get("_covered_by", {}).items():
            if not s.startswith("_"):
                self.covered_by.setdefault(s, set()).update(covering)
        self._regex = re.compile(_trie_regex(self._lookup))
        self._find = lru_cache(maxsize=100_000)(self._find_uncached)

    def _symptom(self, matched: str):
        key = " ".join(re.split(r"[\s-]+", matched))
        if key in self._lookup:
            return self._lookup[key]
        for suffix in SUFFIXES:
            if key.endswith(suffix) and key[: -len(suffix)] in self._lookup:
                return self._lookup[key[: -len(suffix)]]
        return None

    def _find_uncached(self, low: str) -> tuple:
        return tuple((m.start(), m.end(), s) for m in self._regex.finditer(low) if (s := self._symptom(m.group())))

    @classmethod
    def load(cls, path=None, symptoms=None) -> "Lexicon":
        return cls(json.loads(Path(path or DEFAULT_PATH).read_text(encoding="utf-8")), symptoms)

    def lay(self, symptom: str) -> str:
        return self.phrases.get(symptom, [symptom.lower()])[0]

    def find(self, text: str) -> list:
        """Leftmost-longest symptom mentions as (start, end, symptom), offsets into the simplified text."""
        return list(self._find(_simplify(text)))

    def on_card(self, symptom: str, card_symptoms) -> bool:
        """True if the card has this symptom, or one that everyday speech does not tell apart from it."""
        return symptom in card_symptoms or not self.covered_by.get(symptom, set()).isdisjoint(card_symptoms)

    def mentions(self, text: str) -> list:
        """Mentions in a patient statement as (symptom, polarity), polarity True / False / None (unsure)."""
        low = _simplify(text)
        lead = LEADING_ANSWER.match(low)
        floor = lead.end() if lead else 0
        breaks = [m.end() for m in CLAUSE_BREAK.finditer(low)]
        out = []
        for start, _, symptom in self.find(low):
            clause_start = max([floor] + [b for b in breaks if b <= start])
            prefix = low[clause_start:start]
            if UNCERTAIN.search(prefix):
                out.append((symptom, None))
            else:
                out.append((symptom, not NEGATION.search(prefix)))
        return out


def answer_polarity(text: str):
    """True for a yes-type answer, False for a no-type answer, None otherwise."""
    low = _normalise(text)
    if NO_ANSWER.match(low):
        return False
    if YES_ANSWER.match(low):
        return True
    return None


def revealed_symptoms(turns: list, lexicon: Lexicon) -> dict:
    """{symptom: True (has it) / False (denied)} from a list of {"role", "text"} turns.

    A symptom the patient names keeps its latest stated polarity. A bare yes/no answer only fills in symptoms
    nothing has been said about yet: after "I've had a rash on my lip", "Any rash elsewhere?" -> "No, just
    the lip" does not take the rash back."""
    revealed, asked = {}, []
    for t in turns:
        if t["role"] == "doctor":
            asked = list(dict.fromkeys(s for _, _, s in lexicon.find(t["text"])))
            continue
        for symptom, polarity in lexicon.mentions(t["text"]):
            if polarity is not None:
                revealed[symptom] = polarity
        polarity = answer_polarity(t["text"])
        for symptom in asked:
            if symptom not in revealed and (polarity is False or (polarity is True and len(asked) == 1)):
                revealed[symptom] = polarity
        asked = []
    return revealed

### Disease names for the leakage check
`EXTRA_SYNONYMS` and `named_diseases` are copied from `simclinic/cards.py`. A disease's names are:
- its full name
- any part in brackets ("GERD")
- the name without the bracketed part
- its extra synonyms

Shiv's filter also used Symcat's own synonym lists. Those need `simclinic/symcat.py` and the Symcat CSV, which are not in the repo, so they are left out here. As in his filter, names that are also symptoms (for example "Anxiety") are not counted as leakage.

In [9]:
# ---- copied verbatim from simclinic/cards.py ----
EXTRA_SYNONYMS = {
    "Common cold": ["Upper respiratory tract infection", "Upper respiratory infection", "URTI", "URI"],
    "Urinary tract infection": ["UTI"],
    "Acute otitis media": ["Middle ear infection", "Ear infection"],
    "Strep throat": ["Strep"],
    "Seasonal allergies (hay fever)": ["Allergic rhinitis"],
    "Infectious gastroenteritis": ["Gastroenteritis", "Viral gastroenteritis", "Stomach flu"],
    "Gastroesophageal reflux disease (GERD)": ["Acid reflux", "GORD"],
    "Irritable bowel syndrome": ["IBS"],
    "Tension headache": ["Tension-type headache"],
    "Primary insomnia": ["Insomnia"],
    "Fungal infection of the skin": ["Ringworm", "Tinea corporis"],
}


def named_diseases(text: str, names: list) -> list:
    """The names in `names` that appear in text as words ("flu-like" describes a symptom, not the flu)."""
    low = text.lower()
    return [n for n in names if len(n) >= 3 and re.search(rf"\b{re.escape(n.lower())}(?:s|es)?\b(?![- ]like)", low)]



def disease_names(name):
    """cards.disease_names without Symcat's synonym lists."""
    names = [name, *re.findall(r"\(([^)]+)\)", name), re.sub(r"\s*\([^)]*\)", "", name)]
    return list(dict.fromkeys([*names, *EXTRA_SYNONYMS.get(name, [])]))

### Run the evaluation
- **Fidelity:** of the symptoms the patient says it has (Shiv's `revealed_symptoms`), the share that are on the card under `on_card`. Mentions are pooled over all dialogues.
- **Coverage:** the share of the card's symptoms the patient revealed.
- **Leakage:** the share of dialogues where any patient turn names the disease.

In [10]:
if "eval" in STEPS:
    all_cards = {c["card_id"]: c for c in read_jsonl(CARDS_JSONL)}
    split_ids = sorted(c for c, card in all_cards.items() if card["split"] == EVAL_SPLIT)
    n = N_EVAL_CARDS[EVAL_SPLIT]
    card_ids = sorted(random.Random(SEED).sample(split_ids, n)) if n and n < len(split_ids) else split_ids
    eval_cards = [all_cards[c] for c in card_ids]
    lexicon = Lexicon.load(symptoms=sorted({s for c in all_cards.values() for s in c["symptoms"]}))
    leak_names = {d: [x for x in disease_names(d) if [(a, b) for a, b, _ in lexicon.find(x)] != [(0, len(x))]]
                  for d in {c["disease"] for c in all_cards.values()}}

    runs = read_runs()
    stage2 = runs[runs["stage"] == 2] if len(runs) else runs
    if EVAL_SPLIT == "dev" or not len(stage2):
        chosen = list(stage2["run"]) if len(stage2) else []
    else:
        best = list(stage2[~stage2["run"].str.endswith("nostage1")].sort_values("best_dev_loss").groupby("model")["run"].first())
        chosen = (FINAL_RUNS or best) + [r for r in stage2["run"] if r.endswith("nostage1")]
    systems = {"gpt2 prompted": ("gpt2", True)}
    if not QUICK:
        systems["gpt2-medium prompted"] = ("gpt2-medium", True)
    systems |= {r: (MODELS_DIR / r, False) for r in chosen if finished(r)}
    print(f"{EVAL_SPLIT}: {len(eval_cards)} cards x {len(QUESTIONS)} questions | systems: {list(systems)}")

    def score(card, turns):
        revealed = revealed_symptoms(turns, lexicon)
        has = [s for s, present in revealed.items() if present]
        on = [s for s in has if lexicon.on_card(s, card["symptoms"])]
        replies = [t["text"] for t in turns if t["role"] == "patient"]
        return {"mentioned": len(has), "on_card": len(on), "off_card_symptoms": "; ".join(sorted(set(has) - set(on))),
                "coverage": sum(bool(revealed.get(s)) for s in card["symptoms"]) / len(card["symptoms"]),
                "leaked": bool(named_diseases(" ".join(replies), leak_names[card["disease"]])),
                "empty_replies": sum(not r for r in replies),
                "reply_words": float(np.mean([len(r.split()) for r in replies]))}

    def render(turns):
        return "\n".join(("Doctor: " if t["role"] == "doctor" else "Patient: ") + t["text"] for t in turns)

    records = []
    for name, (source, prompted) in systems.items():
        t0 = time.time()
        torch.manual_seed(SEED)
        system = Patient(source, prompted)
        for card, turns in zip(eval_cards, interview(system, eval_cards)):
            records.append({"system": name, "card_id": card["card_id"], "disease": card["disease"],
                            **score(card, turns), "transcript": render(turns)})
        del system
        gc.collect()
        print(f"{name}: {(time.time() - t0) / 60:.1f} min")
    per_dialogue = pd.DataFrame(records)
    per_dialogue.to_csv(RESULTS_DIR / f"patient_eval_{EVAL_SPLIT}.csv", index=False)
    r = per_dialogue.iloc[-1]
    print(f"\nexample ({r.system}, {r.disease}; card: {'; '.join(lay(s) for s in all_cards[r.card_id]['symptoms'])}):\n{r.transcript}")

test: 201 cards x 16 questions | systems: ['gpt2 prompted', 'gpt2-medium prompted', 'stage2-gpt2-lr0.0001', 'stage2-gpt2-medium-lr5e-05', 'stage2-gpt2-lr5e-05-nostage1', 'stage2-gpt2-medium-lr5e-05-nostage1']


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7309.21it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1025 > 1024). Running this sequence through the model will result in indexing errors


gpt2 prompted: 1.7 min


Loading weights: 100%|██████████| 292/292 [00:00<00:00, 314.96it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1054 > 1024). Running this sequence through the model will result in indexing errors


gpt2-medium prompted: 5.1 min


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 710.99it/s]


... (3 outputs left out)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 724.40it/s]


stage2-gpt2-lr5e-05-nostage1: 0.7 min


Loading weights: 100%|██████████| 292/292 [00:00<00:00, 511.72it/s]


stage2-gpt2-medium-lr5e-05-nostage1: 1.9 min

example (stage2-gpt2-medium-lr5e-05-nostage1, Viral exanthem; card: a rash; a fever; skin that looks abnormal; itchy skin):
Doctor: What brings you in today?
Patient: Rash. Fever too.
Doctor: Where exactly do you feel it?
Patient: All over.
Doctor: When did it start, and did it come on suddenly or gradually?
Patient: Yesterday.
Doctor: Can you describe what it feels like?
Patient: Itchy.
Doctor: Does it spread anywhere else?
Patient: No.
Doctor: Have you noticed any other symptoms?
Patient: Swelling on my neck.
Doctor: Is it there all the time, or does it come and go?
Patient: All the time.
Doctor: Does anything make it better or worse?
Patient: Nothing really.
Doctor: On a scale of 0 to 10, how bad is it?
Patient: About 7.
Doctor: Have you had anything like this before?
Patient: No.
Doctor: Are you taking any medication, or have you tried anything for it?
Patient: No.
Doctor: Do you have any allergies?
Patient: No.
Doctor: Do you have any 

### Rater sheet for staying in character
The sheet is blinded: its rows are shuffled, and the system names are kept in `rater_key_<split>.csv`.
- **dev:** the best Stage 2 run per size by dev loss, on all 50 cards
- **test:** the prompted `gpt2` and the final runs, on `RATER_CARDS` random cards each

Each rater fills in their own column with a score from 1 to 5:

| score | the patient... |
|---|---|
| 5 | stays consistent with the card and persona in every turn, answers the question asked, sounds like a real patient |
| 4 | one small slip (a vague or off-topic answer), otherwise consistent |
| 3 | several slips, or one contradiction of the card (a symptom it denied earlier, a different onset) |
| 2 | often off-topic or contradictory, but still recognisably a patient |
| 1 | breaks the role (talks like a doctor or a website, or about something unrelated) or is incoherent |

A sheet that already holds scores is never overwritten.

In [11]:
if "eval" in STEPS:
    sheet_path, key_path = RESULTS_DIR / f"rater_sheet_{EVAL_SPLIT}.csv", RESULTS_DIR / f"rater_key_{EVAL_SPLIT}.csv"
    if EVAL_SPLIT == "dev":
        rate = list(stage2.sort_values("best_dev_loss").groupby("model")["run"].first()) if len(stage2) else []
        rated = per_dialogue[per_dialogue["system"].isin(rate)]
    else:
        rate = ["gpt2 prompted"] + [s for s in chosen if not s.endswith("nostage1")]
        sample = random.Random(SEED).sample(card_ids, min(RATER_CARDS, len(card_ids)))
        rated = per_dialogue[per_dialogue["system"].isin(rate) & per_dialogue["card_id"].isin(sample)]

    def describe(card):
        p = ", ".join(f"{k.replace('_', ' ')}: {v}" for k, v in card["persona"].items())
        return f"{card['disease']} | symptoms: {'; '.join(lay(s) for s in card['symptoms'])} | {p}"

    if sheet_path.exists() and pd.read_csv(sheet_path)[RATERS].notna().any().any():
        print(f"{sheet_path.name} already has scores, so it was not overwritten")
    elif len(rated):
        order = rated.sample(frac=1, random_state=SEED).reset_index(drop=True)
        order["sheet_id"] = [f"t{i:04d}" for i in range(len(order))]
        pd.DataFrame({"sheet_id": order["sheet_id"], "card": order["card_id"].map(lambda c: describe(all_cards[c])),
                      "transcript": order["transcript"], **{r: np.nan for r in RATERS}, "notes": ""}).to_csv(sheet_path, index=False)
        order[["sheet_id", "system", "card_id"]].to_csv(key_path, index=False)
        print(f"wrote {sheet_path.name}: {len(order)} transcripts from {rate}")

wrote rater_sheet_test.csv: 90 transcripts from ['gpt2 prompted', 'stage2-gpt2-lr0.0001', 'stage2-gpt2-medium-lr5e-05']


### Summary
- **Fidelity** is pooled over all mentions: on-card mentions divided by all mentions.
- **In character** is the mean of the two raters. It appears once the sheet has scores, together with how far the raters agree.

On dev, the selection table adds each Stage 2 run's dev loss.

In [12]:
if "eval" in STEPS:
    summary = per_dialogue.groupby("system", sort=False).agg(
        dialogues=("card_id", "size"), mentioned=("mentioned", "sum"), on_card=("on_card", "sum"),
        coverage=("coverage", "mean"), leakage=("leaked", "mean"),
        empty_replies=("empty_replies", "sum"), reply_words=("reply_words", "mean"))
    summary.insert(3, "symptom_fidelity", summary["on_card"] / summary["mentioned"].replace(0, np.nan))
    if sheet_path.exists() and key_path.exists():
        scored = pd.read_csv(sheet_path).merge(pd.read_csv(key_path), on="sheet_id").dropna(subset=RATERS)
        if len(scored):
            summary["in_character"] = scored.groupby("system")[RATERS].mean().mean(axis=1)
            diff = (scored[RATERS[0]] - scored[RATERS[1]]).abs()
            print(f"raters: {len(scored)} transcripts scored by both; mean |difference| {diff.mean():.2f}, within 1 point {np.mean(diff <= 1):.0%}")
        else:
            print("no rater scores yet")
    if len(stage2):
        summary = summary.join(stage2.set_index("run")[["best_dev_loss"]].rename(columns={"best_dev_loss": "stage2_dev_loss"}))
    summary.round(3).to_csv(RESULTS_DIR / f"patient_eval_summary_{EVAL_SPLIT}.csv")
    display(summary.round(3))

no rater scores yet


,dialogues,mentioned,on_card,symptom_fidelity,coverage,leakage,empty_replies,reply_words,stage2_dev_loss
system,,,,,,,,,
gpt2 prompted,201,184,68,0.370,0.076,0.124,0,9.093,NaN
gpt2-medium prompted,201,361,142,0.393,0.151,0.303,0,11.173,NaN
stage2-gpt2-lr0.0001,201,633,598,0.945,0.678,0.010,0,9.922,0.888
stage2-gpt2-medium-lr5e-05,201,672,626,0.932,0.713,0.025,0,10.342,0.824
stage2-gpt2-lr5e-05-nostage1,201,660,602,0.912,0.673,0.005,0,9.158,0.932
stage2-gpt2-medium-lr5e-05-nostage1,201,683,639,0.936,0.710,0.015,0,9.367,0.836


## 6. Check against the app's loader (run locally, not needed on the cluster)
The app builds prompts with `simclinic/patient.py`. This cell checks that it builds exactly the strings this notebook trained on. It is skipped when `simclinic/` is not there.

In [13]:
if not (ROOT / "simclinic" / "patient.py").exists():
    print("simclinic/patient.py not found here, check skipped")
else:
    import sys
    sys.path.insert(0, str(ROOT))
    from simclinic import patient as app
    dialogues = read_jsonl(DIALOGUES_JSONL)
    tok = AutoTokenizer.from_pretrained(str(PREP_DIR / "tokenizer"))
    for d in dialogues:
        card = {"disease": d["disease"], "symptoms": d["symptoms"], "persona": d["persona"]}
        assert app.format_dialogue(d["turns"], card) == format_dialogue(d["turns"], card), d["dialogue_id"]
        history = d["turns"][:3]
        assert app.build_prompt(card, history, tok, 1024) == build_prompt(card, history, tok, 1024), d["dialogue_id"]
    assert app.SPECIAL_TOKENS == SPECIAL_TOKENS
    print(f"simclinic/patient.py builds the same strings on all {len(dialogues):,} dialogues")

simclinic/patient.py not found here, check skipped
